In [1]:
import pandas as pd


df = pd.read_csv('../csv/50area_dummy1_232,000.csv', parse_dates=['마지막충전종료시간', '연결시작시간', '충전시작시간', '충전종료시간', '연결종료시간', '출발예상시간'])

In [2]:
df.head()

,충전소위치,충전기이름,충전기타입,충전기상태,충전기용량,방전지원여부,예약충전,충전량(kWh),요청충전량(kWh),방전량(kWh),...,마지막충전종료시간,연결시작시간,충전시작시간,충전종료시간,연결종료시간,출발예상시간,베터리용량(kWh),시작베터리%,종료베터리%,연비(Wh/km)
0,CSCS2015,0,FC,Available,50,n,n,12.84,12.62,0,...,2024-07-28 11:46:02.688864,2024-07-30 00:23:02.688864,2024-07-30 00:40:02.688864,2024-07-30 00:50:02.688864,2024-07-30 01:10:54.688864,2024-07-30 00:51:02.688864,100,33.0,5.5,5.98
1,CSCS2015,0,FC,Available,50,n,n,17.64,17.81,0,...,2024-07-30 01:10:54.688864,2024-07-31 19:16:54.688864,2024-07-31 19:35:54.688864,2024-07-31 19:51:54.688864,2024-07-31 21:17:14.688864,2024-07-31 20:05:54.688864,90,17.7,17.8,5.27
2,CSCS2015,0,FC,Available,50,n,n,40.58,41.30,0,...,2024-07-31 21:17:14.688864,2024-08-03 15:56:14.688864,2024-08-03 16:15:14.688864,2024-08-03 16:52:14.688864,2024-08-03 17:17:54.688864,2024-08-03 16:55:14.688864,90,14.0,48.7,6.09
3,CSCS2015,0,FC,Available,50,n,y,21.27,21.08,0,...,2024-08-03 17:17:54.688864,2024-08-04 00:46:54.688864,2024-08-04 01:13:54.688864,2024-08-04 01:31:54.688864,2024-08-04 01:42:36.688864,2024-08-04 01:46:54.688864,100,15.6,22.5,4.83
4,CSCS2015,0,FC,Available,50,n,n,9.17,9.30,0,...,2024-08-04 01:42:36.688864,2024-08-06 14:30:36.688864,2024-08-06 14:37:36.688864,2024-08-06 14:46:36.688864,2024-08-06 15:02:34.688864,2024-08-06 14:40:36.688864,75,16.6,10.8,5.35


In [3]:
import pandas as pd
from datetime import datetime
import numpy as np
from pandas.api.types import is_datetime64_any_dtype

df = pd.read_csv('../csv/50area_dummy1_232,000.csv')

# [1] 시간 필드가 datetime 형인지 확인/변환
time_cols = [
    '마지막충전종료시간', '연결시작시간', '충전시작시간',
    '충전종료시간', '연결종료시간', '출발예상시간'
]
for c in time_cols:
    if not is_datetime64_any_dtype(df[c]):
        df[c] = pd.to_datetime(df[c])

# [2] 추가 Feature Engineering 파생 컬럼 (질문 예시 기준)
df['요일'] = df['연결시작시간'].dt.weekday
df['사용하지않은시간'] = df['연결시작시간'] - df['마지막충전종료시간']
df['사용자예상충전소이용시간'] = df['출발예상시간'] - df['연결시작시간']
df['사용자예상충전소요시간차이'] = df['출발예상시간'] - df['충전종료시간']
df['실제총충전소이용시간'] = df['연결종료시간'] - df['연결시작시간']
df['실제충전시간'] = df['충전종료시간'] - df['충전시작시간']
df['충전시작소요시간'] = df['충전시작시간'] - df['연결시작시간']
df['충전완료후출발소요시간'] = df['연결종료시간'] - df['충전종료시간']
df['실제이용시간과예상출발시간차이'] = df['연결종료시간'] - df['출발예상시간']
df['요청충전량차이'] = df['요청충전량(kWh)'] - df['충전량(kWh)']
# 사용자가 인식하는 충전에 소요되는 시간
df['충전량당시간이해관계'] = df['사용자예상충전소이용시간'].dt.total_seconds() / 3600 / df['요청충전량(kWh)'].replace({0: np.nan})
df['이용시간당전력사용량'] = df['충전량(kWh)'] / (df['실제충전시간'].dt.total_seconds() / 3600).replace({0: np.nan})

# ... 이하 완벽히 동일하게 진행 (생략) ...

# 실제 (충전 총량)/(실제 충전시간(h))
df['이용시간당전력사용량'] = df['충전량(kWh)'] / (
    df['실제충전시간'].dt.total_seconds() / 3600
)

# [3] 타겟 컬럼 사전 매핑 (원본명 → 타겟명)
col_map = {
    'station_location': '충전소위치',
    'evse_name': '충전기이름',
    'evse_type': '충전기타입',
    'supports_discharge': '방전지원여부',
    'scheduled_charge': '예약충전',
    'delivered_kwh': '충전량(kWh)',
    'requested_kwh': '요청충전량(kWh)',
    'kwh_per_usage_time': '연비(Wh/km)',
    'last_charge_end_time': '마지막충전종료시간',
    'connection_start_time': '연결시작시간',
    'charging_start_time': '충전시작시간',
    'charging_end_time': '충전종료시간',
    'connection_end_time': '연결종료시간',
    'expected_departure_time': '출발예상시간',
    # 추가 Feature
    'weekday':'요일',
    'idle_time': '사용하지않은시간',
    'expected_usage_duration':'사용자예상충전소이용시간',
    'expected_time_diff':'사용자예상충전소요시간차이',
    'actual_usage_duration':'실제총충전소이용시간',
    'actual_charging_duration':'실제충전시간',
    'start_delay_duration':'충전시작소요시간',
    'post_charge_departure_delay':'충전완료후출발소요시간',
    'usage_departure_time_diff':'실제이용시간과예상출발시간차이',
    'kwh_request_diff':'요청충전량차이',
    'duration_per_kwh':'충전량당시간이해관계',
    'kwh_per_usage_time_2':'이용시간당전력사용량'
}

# [4] unixtimestamp 변환 함수
def iso2unixts(dt): 
    if pd.isna(dt): return ""
    return int(dt.timestamp())

# [5] 파생 및 결측/엔지니어링 컬럼 만들기
out_df = pd.DataFrame()

# 타겟 명세 필드(원하는 최종 컬럼 순서)
final_cols = [
    'last_charge_end_time_ts',
    'connection_start_time_ts',
    'charging_start_time_ts',
    'charging_start_time_missing',
    'charging_end_time_ts',
    'charging_end_time_missing',
    'connection_end_time_ts',
    'expected_departure_time_ts',
    'expected_departure_time_missing',
    'idle_time_ts',
    'expected_usage_duration_ts',
    'expected_usage_duration_missing',
    'expected_time_diff_ts',
    'expected_time_diff_missing',
    'actual_usage_duration_ts',
    'actual_charging_duration_ts',
    'actual_charging_duration_missing',
    'start_delay_duration_ts',
    'start_delay_duration_missing',
    'post_charge_departure_delay_ts',
    'post_charge_departure_delay_missing',
    'usage_departure_time_diff_ts',
    'usage_departure_time_diff_missing',
    'duration_per_kwh_ts',
    'duration_per_kwh_missing',
    'delivered_kwh',
    'requested_kwh',
    'kwh_request_diff',
    'kwh_per_usage_time',
    'station_location',
    'evse_name',
    'evse_type',
    'supports_discharge',
    'scheduled_charge',
    'weekday',
    'usage_departure_range',
    'post_charge_departure_range',
    'cluster',
    # 추가 피쳐
    'idle_time', 'expected_usage_duration', 'expected_time_diff',
    'actual_usage_duration', 'actual_charging_duration',
    'start_delay_duration', 'post_charge_departure_delay',
    'usage_departure_time_diff', 'duration_per_kwh','kwh_per_usage_time_2'
]

# 각종 unixts 컬럼
for tcol, scol in [
    ('last_charge_end_time_ts', '마지막충전종료시간'),
    ('connection_start_time_ts', '연결시작시간'),
    ('charging_start_time_ts', '충전시작시간'),
    ('charging_end_time_ts', '충전종료시간'),
    ('connection_end_time_ts', '연결종료시간'),
    ('expected_departure_time_ts', '출발예상시간')
]:
    out_df[tcol] = df[scol].apply(iso2unixts)

def safe_diff(a, b):
    if a == "" or b == "": return ""
    return int(b) - int(a)

def dt2ts(series):
    # datetime.timedelta 값을 초 단위 int로 변환
    return series.apply(lambda x: "" if pd.isna(x) else int(x.total_seconds()))

def datetime_to_unixts(x):
    if pd.isna(x): return np.nan
    return int(x.timestamp())

def timedelta_to_seconds(x):
    if pd.isna(x): return np.nan
    return int(x.total_seconds())

# 결측
out_df['expected_departure_time_missing'] = df['출발예상시간'].apply(lambda x: int(x.timestamp()) if pd.notna(x) else np.nan)
out_df['expected_usage_duration_missing'] = df['사용자예상충전소이용시간'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['expected_time_diff_missing'] = df['사용자예상충전소요시간차이'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['actual_usage_duration_missing'] = df['실제총충전소이용시간'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['actual_charging_duration_missing'] = df['실제충전시간'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)


# datetime 컬럼 → unixtimestamp(초)
out_df['expected_departure_time_ts'] = df['출발예상시간'].apply(datetime_to_unixts)

# 기간 컬럼 → 초
out_df['idle_time_ts'] = df['사용하지않은시간'].apply(timedelta_to_seconds)
out_df['expected_usage_duration_ts'] = df['사용자예상충전소이용시간'].apply(timedelta_to_seconds)
out_df['expected_time_diff_ts'] = df['사용자예상충전소요시간차이'].apply(timedelta_to_seconds)
out_df['actual_usage_duration_ts'] = df['실제총충전소이용시간'].apply(timedelta_to_seconds)
out_df['actual_charging_duration_ts'] = df['실제충전시간'].apply(timedelta_to_seconds)

# 기타 duration
out_df['start_delay_duration_ts'] = dt2ts(df['충전시작소요시간'])
out_df['start_delay_duration_missing'] = df['충전시작소요시간'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['post_charge_departure_delay_ts'] = dt2ts(df['충전완료후출발소요시간'])
out_df['post_charge_departure_delay_missing'] = df['충전완료후출발소요시간'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['usage_departure_time_diff_ts'] = dt2ts(df['실제이용시간과예상출발시간차이'])
out_df['usage_departure_time_diff_missing'] = df['실제이용시간과예상출발시간차이'].apply(lambda x: int(x.total_seconds()) if pd.notna(x) else np.nan)
out_df['delivered_kwh'] = pd.to_numeric(df['충전량(kWh)'], errors='coerce')
out_df['duration_per_kwh_missing'] = df['충전량(kWh)']==0
out_df['kwh_per_usage_time_missing'] = df['연비(Wh/km)'].isna()

# 구간 설정 예시(0:0~30분, 1:30~60분...)
def usage_range(x):
    if x=="" or pd.isna(x): return -1
    return int(x)//1800

out_df['usage_departure_range'] = out_df['expected_usage_duration_ts'].apply(usage_range)
out_df['post_charge_departure_range'] = out_df['post_charge_departure_delay_ts'].apply(usage_range)

# 클러스터 기본값 설정
out_df['cluster'] = 0

# 기타 문자/수치 컬럼 복사
for tcol, scol in col_map.items():
    if tcol in ['delivered_kwh','requested_kwh','kwh_per_usage_time']:
        out_df[tcol] = df[scol]
    elif tcol in out_df.columns:
        pass # 이미 있음 (시계열)
    else:
        out_df[tcol] = df[scol]

# 추가 피쳐도 복사
for tcol, scol in col_map.items():
    try:
        out_df[scol] = df[scol]
    except: pass

# kwh diff
out_df['kwh_request_diff'] = df['요청충전량(kWh)']-df['충전량(kWh)']

# columns 맞추기
for col in final_cols:
    if col not in out_df.columns:
        # 결측 예시
        if 'missing' in col: out_df[col] = False
        elif col.endswith('_range'): out_df[col] = -1
        else: out_df[col] = ""
final_cols = [
    'last_charge_end_time_ts','connection_start_time_ts','charging_start_time_ts','charging_start_time_missing','charging_end_time_ts','charging_end_time_missing','connection_end_time_ts','expected_departure_time_ts','expected_departure_time_missing','idle_time_ts','expected_usage_duration_ts','expected_usage_duration_missing','expected_time_diff_ts','expected_time_diff_missing','actual_usage_duration_ts','actual_charging_duration_ts','actual_charging_duration_missing','start_delay_duration_ts','start_delay_duration_missing','post_charge_departure_delay_ts','post_charge_departure_delay_missing','usage_departure_time_diff_ts','usage_departure_time_diff_missing','duration_per_kwh_ts','duration_per_kwh_missing','delivered_kwh','requested_kwh','kwh_request_diff','kwh_per_usage_time','kwh_per_usage_time_missing','station_location','evse_name','evse_type','supports_discharge','scheduled_charge','weekday','usage_departure_range','post_charge_departure_range','cluster'
]

out_df = out_df[final_cols]

# 확인 및 저장
print(out_df.head())
out_df.to_csv('../csv/50area_dummy_processed.csv', index=False, header=True)


   last_charge_end_time_ts  connection_start_time_ts charging_start_time_ts  \
0               1722167162                1722298982             1722300002   
1               1722301854                1722453414             1722454554   
2               1722460634                1722700574             1722701714   
3               1722705474                1722732414             1722734034   
4               1722735756                1722954636             1722955056   

   charging_start_time_missing charging_end_time_ts  \
0                        False           1722300602   
1                        False           1722455514   
2                        False           1722703934   
3                        False           1722735114   
4                        False           1722955596   

   charging_end_time_missing  connection_end_time_ts  \
0                      False              1722301854   
1                      False              1722460634   
2                      Fal